# Supplementary Figure S7: Immune Checkpoint Receptor Expression by Ecotype
**Pediatric Glioma Immune Ecotype Framework**

Analysis: log₂(TPM+1) expression of 8 immune checkpoint genes across 3 ecotypes (n=349).
Statistics: Kruskal-Wallis with Dunn's post-hoc (Bonferroni correction).

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

# Load data
tpm = pd.read_csv('output/tpm_for_cibersortx.tsv', sep='\t', index_col=0)
eco = pd.read_csv('output/ecotype_assignment_k3_annotated.tsv')
print(f'TPM matrix: {tpm.shape}, Ecotype assignments: {len(eco)} samples')

In [ ]:
# Extract checkpoint genes and transform
genes = ['HAVCR2','LAG3','PDCD1','CD274','CTLA4','TIGIT','IDO1','SIGLEC15']
gene_labels = {'HAVCR2':'TIM-3\n(HAVCR2)','LAG3':'LAG-3\n(LAG3)','PDCD1':'PD-1\n(PDCD1)',
               'CD274':'PD-L1\n(CD274)','CTLA4':'CTLA-4\n(CTLA4)','TIGIT':'TIGIT',
               'IDO1':'IDO1','SIGLEC15':'SIGLEC15'}

samples = eco['Kids_First_Biospecimen_ID'].tolist()
expr = tpm.loc[genes, samples].T
expr = np.log2(expr + 1)
expr.index.name = 'Kids_First_Biospecimen_ID'
df = expr.reset_index().merge(eco[['Kids_First_Biospecimen_ID','ecotype']], on='Kids_First_Biospecimen_ID')
print(f'Merged: {len(df)} samples')
df['ecotype'].value_counts()

In [ ]:
# Kruskal-Wallis tests
ecotype_order = ['Inflamed','Intermediate','Immune-desert']
print(f'{"Gene":25s} {"H-stat":>8s} {"p-value":>12s} {"Infl median":>12s} {"Mye median":>12s} {"Des median":>12s}')
print('-'*85)
for g in genes:
    groups = [df.loc[df['ecotype']==e, g].values for e in ecotype_order]
    stat, p = stats.kruskal(*groups)
    meds = [df.loc[df['ecotype']==e, g].median() for e in ecotype_order]
    print(f'{gene_labels[g].replace(chr(10)," "):25s} {stat:8.1f} {p:12.2e} {meds[0]:12.2f} {meds[1]:12.2f} {meds[2]:12.2f}')

In [ ]:
# Dunn's post-hoc test
def dunn_test(df, gene, groups, ecotype_col='ecotype'):
    n_total = len(df)
    ranked = df[gene].rank()
    results = {}
    for g1, g2 in combinations(groups, 2):
        n1, n2 = (df[ecotype_col]==g1).sum(), (df[ecotype_col]==g2).sum()
        r1, r2 = ranked[df[ecotype_col]==g1].mean(), ranked[df[ecotype_col]==g2].mean()
        se = np.sqrt((n_total*(n_total+1)/12) * (1/n1 + 1/n2))
        z = abs(r1 - r2) / se
        p = min(2 * (1 - stats.norm.cdf(z)) * 3, 1.0)  # Bonferroni
        results[(g1, g2)] = p
    return results

for gene in genes:
    dunn = dunn_test(df, gene, ecotype_order)
    parts = [f'{g1[:4]}-{g2[:4]}: p={p:.3f}{"***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else " ns"))}'
             for (g1,g2), p in dunn.items()]
    print(f'{gene:10s}  {", ".join(parts)}')

In [ ]:
# Publication-quality figure
colors = {'Inflamed':'#E64B35','Intermediate':'#4DBBD5','Immune-desert':'#91D1C2'}

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, gene in enumerate(genes):
    ax = axes[idx]
    data = [df.loc[df['ecotype']==e, gene].values for e in ecotype_order]
    bp = ax.boxplot(data, positions=[1,2,3], widths=0.6, patch_artist=True,
                    showfliers=False, medianprops=dict(color='black', linewidth=1.5),
                    whiskerprops=dict(linewidth=1), capprops=dict(linewidth=1))
    for patch, eco in zip(bp['boxes'], ecotype_order):
        patch.set_facecolor(colors[eco])
        patch.set_alpha(0.7)
        patch.set_edgecolor('black')
        patch.set_linewidth(1)
    for i, eco in enumerate(ecotype_order):
        vals = df.loc[df['ecotype']==eco, gene].values
        jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
        ax.scatter(np.full_like(vals, i+1) + jitter, vals, c=colors[eco], s=6, alpha=0.3, edgecolors='none', zorder=3)
    kw_stat, kw_p = stats.kruskal(*data)
    dunn = dunn_test(df, gene, ecotype_order)
    y_max = max(d.max() for d in data)
    y_range = y_max - min(d.min() for d in data)
    sig_offset = 0
    for (x1, x2), pair_key in zip([(1,2),(2,3),(1,3)],
            [(ecotype_order[0],ecotype_order[1]),(ecotype_order[1],ecotype_order[2]),(ecotype_order[0],ecotype_order[2])]):
        p_val = dunn[pair_key]
        if p_val < 0.05:
            y = y_max + y_range * (0.08 + sig_offset * 0.12)
            ax.plot([x1,x1,x2,x2], [y-y_range*0.02,y,y,y-y_range*0.02], lw=0.8, c='black')
            sig_text = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else '*')
            ax.text((x1+x2)/2, y, sig_text, ha='center', va='bottom', fontsize=8)
            sig_offset += 1
    p_str = f'p = {kw_p:.1e}' if kw_p < 0.001 else f'p = {kw_p:.3f}'
    ax.set_title(f'{gene_labels[gene]}\nKW {p_str}', fontsize=10, fontweight='bold')
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(['Infl.','Mye.','Des.'], fontsize=8)
    ax.set_ylabel('log₂(TPM + 1)' if idx % 4 == 0 else '', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(labelsize=8)

legend_elements = [Patch(facecolor=colors[e], edgecolor='black', alpha=0.7, label=e) for e in ecotype_order]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle('Immune Checkpoint Receptor Expression by Immune Ecotype\n(n = 349, Main Cohort)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig('output/figs_300dpi/Supplementary_Figure_S7_checkpoint_expression.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('output/figs_300dpi/Supplementary_Figure_S7_checkpoint_expression.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: Supplementary_Figure_S7_checkpoint_expression.png/.pdf (300 dpi)')